# Contract Details

Adapted from [ib_async contract_details](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates requesting contract metadata for stocks.

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import EWrapper, EClient, Contract, ContractDetails

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.details = {}  # req_id -> list[ContractDetails]
        self.done = {}     # req_id -> Event
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.connected.set()

    def contract_details(self, req_id, contract_details):
        self.details.setdefault(req_id, []).append(contract_details)

    def contract_details_end(self, req_id):
        if req_id in self.done:
            self.done[req_id].set()

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

def req_contract_details(client, app, req_id, contract, timeout=10):
    """Helper: request contract details and wait for response."""
    app.done[req_id] = threading.Event()
    app.details[req_id] = []
    client.req_contract_details(req_id, contract)
    app.done[req_id].wait(timeout=timeout)
    return app.details.get(req_id, [])

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")

### Request contract details for AAPL

We use the conId for AAPL (265598) to get a unique match:

In [ ]:
aapl = Contract(con_id=265598, symbol="AAPL")
details = req_contract_details(client, app, 1, aapl)

print(f"Got {len(details)} result(s)")
if details:
    cd = details[0]
    print(f"  Symbol: {cd.contract.symbol}")
    print(f"  Long name: {cd.long_name}")
    print(f"  Min tick: {cd.min_tick}")
    print(f"  Exchange: {cd.contract.exchange}")
    print(f"  Valid exchanges: {cd.valid_exchanges}")
    print(f"  Order types: {cd.order_types}")

### Try another contract: TSLA

In [ ]:
tsla = Contract(con_id=76792991, symbol="TSLA")
details = req_contract_details(client, app, 2, tsla)

print(f"Got {len(details)} result(s)")
if details:
    cd = details[0]
    print(f"  {cd.contract.symbol} - {cd.long_name}")
    print(f"  Min tick: {cd.min_tick}")

In [ ]:
client.disconnect()